# Data Shapes: Solutions

Instructor copy. Plain loops for the main tasks, matching how they were set.
Shortcut versions are shown alongside where they help.

In [ ]:
import pandas as pd


def shape_of(thing, depth=0):
    """Describe the shape of any nested structure, from the outside in."""
    pad = "  " * depth
    if isinstance(thing, dict):
        if len(thing) == 0:
            print(f"{pad}dict (empty)")
            return
        first_key = list(thing.keys())[0]
        print(f"{pad}dict with {len(thing)} key(s): {list(thing.keys())[:4]}")
        shape_of(thing[first_key], depth + 1)
    elif isinstance(thing, list):
        if len(thing) == 0:
            print(f"{pad}list (empty)")
            return
        print(f"{pad}list of {len(thing)} item(s)")
        shape_of(thing[0], depth + 1)
    else:
        print(f"{pad}{type(thing).__name__}: {thing!r}")


print("helpers ready")

## Reading the shapes: tasks

In [ ]:
# Task 1
student = {"name": "Ada", "score": 78, "track": "data"}

print(student["name"])
print(student.get("score"))
print(student.get("email", "not provided"))

student["level"] = 300

for key, value in student.items():
    print(f"{key}: {value}")

In [ ]:
students = [
    {"id": 1, "name": "Ada",   "score": 78, "track": "data"},
    {"id": 2, "name": "Tunde", "score": 64, "track": "web"},
    {"id": 3, "name": "Chi",   "score": 91, "track": "data"},
]

# Task 2: every name, then the high scorers
for row in students:
    print(row["name"])

print()
for row in students:
    if row["score"] > 70:
        print(row["name"], row["score"])

In [ ]:
# Task 3: count per track
by_track = {}
for row in students:
    track = row["track"]
    if track in by_track:
        by_track[track] = by_track[track] + 1
    else:
        by_track[track] = 1

print(by_track)

In [ ]:
# Task 4: build the lookup
by_id = {}
for row in students:
    by_id[row["id"]] = row

print(by_id[2]["name"])

In [ ]:
# Task 5: columnar, walked by position
cols = {"name": ["Ada", "Tunde"], "score": [78, 64]}

for i in range(len(cols["name"])):
    print(f"{cols['name'][i]} scored {cols['score'][i]}")

In [ ]:
# Task 6: unpacking in the for line
data = [[1, "Ada", 78], [2, "Tunde", 64]]

for student_id, name, score in data:
    print(name)

In [ ]:
# Task 7: safe nested access
people = [
    {"name": "Ada",   "address": {"city": "Lagos"}},
    {"name": "Tunde"},
    {"name": "Chi",   "address": {}},
]

for person in people:
    city = person.get("address", {}).get("city", "unknown")
    print(f"{person['name']}: {city}")

In [ ]:
response = {
    "status": "ok",
    "data": [
        {"name": "Ada",   "orders": [{"amt": 50}, {"amt": 20}]},
        {"name": "Tunde", "orders": [{"amt": 10}]},
        {"name": "Chi",   "orders": []},
    ],
}

# Task 8: check the envelope, then report
if response["status"] != "ok":
    print("the API reported a problem, stopping")
else:
    for record in response["data"]:
        print(f"{record['name']}: {len(record['orders'])} order(s)")

In [ ]:
# Task 9: flatten
flat = []
for record in response["data"]:
    for order in record["orders"]:
        flat.append({"name": record["name"], "amt": order["amt"]})

for row in flat:
    print(row)

print()
print(f"{len(response['data'])} people became {len(flat)} order rows")
print("Chi has no orders, so the inner loop never ran and she is not in the result.")

In [ ]:
# Task 10: totals per person, with the reset in the right place
for record in response["data"]:
    total = 0                                  # inside the outer loop
    for order in record["orders"]:
        total = total + order["amt"]
    print(f"{record['name']}: {total}")

## Drill: name that shape

| | Shape | Get the value | `pd.DataFrame(thing)` directly? |
|---|---|---|---|
| 1 | dict, one record | `mystery_1["population"]` | No, wrap it: `pd.DataFrame([m1])` |
| 2 | list of dict | `mystery_2[1]["city"]` | Yes |
| 3 | dict of dict | `mystery_3["lagos"]["population"]` | Runs, but wrong. Use `from_dict(orient="index")` |
| 4 | dict of list | `mystery_4["city"][1]` | Yes |
| 5 | list of list | `mystery_5[0][1]` | Yes, but pass `columns=` |
| 6 | dict of list of dict, with a nested list | `mystery_6["results"][0]["districts"][0]["name"]` | No. `json_normalize` with `record_path` |
| 7 | list of dict with a nested dict | `mystery_7[0]["coords"]["lat"]` | Runs, but the dict sits in a cell. Use `json_normalize` |
| 8 | dict of dict of list of dict | `mystery_8["data"]["cities"][0]["name"]` | No. `pd.DataFrame(m8["data"]["cities"])` |

In [ ]:
mystery_1 = {"city": "Lagos", "population": 15000000}
mystery_2 = [{"city": "Lagos"}, {"city": "Abuja"}]
mystery_3 = {"lagos": {"population": 15000000}, "abuja": {"population": 3500000}}
mystery_4 = {"city": ["Lagos", "Abuja"], "population": [15000000, 3500000]}
mystery_5 = [["Lagos", 15000000], ["Abuja", 3500000]]
mystery_6 = {"results": [{"city": "Lagos", "districts": [{"name": "Ikeja"}]}]}
mystery_7 = [{"city": "Lagos", "coords": {"lat": 6.5, "lon": 3.3}}]
mystery_8 = {"ok": True, "data": {"cities": [{"name": "Lagos"}]}}

print(mystery_1["population"])
print(mystery_2[1]["city"])
print(mystery_3["lagos"]["population"])
print(mystery_4["city"][1])
print(mystery_5[0][1])
print(mystery_6["results"][0]["districts"][0]["name"])
print(mystery_7[0]["coords"]["lat"])
print(mystery_8["data"]["cities"][0]["name"])

## Conversions

In [ ]:
rows = [
    {"id": 1, "name": "Ada",   "age": 30},
    {"id": 2, "name": "Tunde", "age": 25},
]

# list of dict -> dict of dict
by_id = {}
for row in rows:
    by_id[row["id"]] = row
print("dict of dict:", by_id)

# dict of dict -> list of dict
back = []
for key, record in by_id.items():
    copy = dict(record)
    copy["id"] = key
    back.append(copy)
print("list of dict:", back)

# list of dict -> dict of list
cols = {}
for row in rows:
    for key, value in row.items():
        if key not in cols:
            cols[key] = []
        cols[key].append(value)
print("dict of list:", cols)

# dict of list -> list of dict
rebuilt = []
for i in range(len(cols["id"])):
    record = {}
    for key in cols:
        record[key] = cols[key][i]
    rebuilt.append(record)
print("rebuilt:     ", rebuilt)
print("round trip matches:", rebuilt == rows)

## The shortcuts: tasks

In [ ]:
students = [
    {"id": 1, "name": "Ada",   "score": 78, "track": "data"},
    {"id": 2, "name": "Tunde", "score": 64, "track": "web"},
    {"id": 3, "name": "Chi",   "score": 91, "track": "data"},
]
names = ["Ada", "Tunde", "Chi"]
columns = ["id", "name", "score"]
values = [1, "Ada", 78]
data = [[1, "Ada", 78], [2, "Tunde", 64]]
cols = {"id": [1, 2], "name": ["Ada", "Tunde"]}

# Task 1: enumerate
for position, name in enumerate(names, start=1):
    print(f"{position}. {name}")

In [ ]:
# Task 2: zip into a record
record = dict(zip(columns, values))
print(record)

In [ ]:
# Task 3: list of list -> list of dict with zip
converted = []
for row in data:
    converted.append(dict(zip(columns, row)))

for item in converted:
    print(item)

In [ ]:
# Task 4: dict of list, walked with zip
for student_id, name in zip(cols["id"], cols["name"]):
    print(student_id, name)

In [ ]:
# Task 5: sort by score, top two
ranked = sorted(students, key=lambda row: row["score"], reverse=True)

for row in ranked[:2]:
    print(row["name"], row["score"])

In [ ]:
# Task 6: max with a key vs max of the values
best = max(students, key=lambda row: row["score"])
print("the record:", best)

scores = []
for row in students:
    scores.append(row["score"])
print("just the number:", max(scores))

In [ ]:
# Task 7: sort by two fields
for row in sorted(students, key=lambda r: (r["track"], r["score"])):
    print(row["track"], row["name"], row["score"])

In [ ]:
# Task 8: any and all
above_50 = []
above_90 = []
for row in students:
    above_50.append(row["score"] > 50)
    above_90.append(row["score"] > 90)

print(f"everyone above 50? {all(above_50)}")
print(f"anyone above 90?   {any(above_90)}")
print(f"how many above 50? {sum(above_50)}")

In [ ]:
# Task 9: map and filter
all_names = list(map(lambda row: row["name"], students))
passed = list(filter(lambda row: row["score"] >= 70, students))

print(all_names)
for row in passed:
    print(row["name"], row["score"])

In [ ]:
# Task 10: map with a built-in, no lambda
raw = ["10", "20", "30"]
numbers = list(map(int, raw))

print(numbers, sum(numbers))

In [ ]:
# Task 11: the flatten, and the honest answer
customers = [
    {"id": 1, "name": "Ada", "orders": [{"amt": 50}, {"amt": 20}]},
    {"id": 2, "name": "Tunde", "orders": [{"amt": 10}]},
]

flat = []
for customer in customers:
    for order in customer["orders"]:
        flat.append({"id": customer["id"], "name": customer["name"], "amt": order["amt"]})

for row in flat:
    print(row)

print()
print("None of map, filter or zip helps here. The loop pairs a PARENT with each of")
print("its CHILDREN, which is not what any of them do. A nested comprehension can")
print("express it, but it is harder to read than the two loops. Leave it as loops.")

## Mini project: full worked solution

In [ ]:
response = {
    "status": "ok",
    "page": 1,
    "total": 5,
    "data": [
        {"id": 101, "name": "Ada",   "city": "Lagos",
         "orders": [{"item": "laptop", "amt": 450000}, {"item": "mouse", "amt": 8000}]},
        {"id": 102, "name": "Tunde", "city": "Abuja",
         "orders": [{"item": "phone", "amt": 180000}]},
        {"id": 103, "name": "Chi",   "city": "Lagos",
         "orders": [{"item": "monitor", "amt": 95000}, {"item": "keyboard", "amt": 15000},
                    {"item": "headset", "amt": 22000}]},
        {"id": 104, "name": "Musa",  "city": "Kano", "orders": []},
        {"id": 105, "name": "Ngozi", "city": "Lagos",
         "orders": [{"item": "laptop", "amt": 520000}]},
    ],
}

# 1. check the envelope
print(f"status: {response['status']}")
if len(response["data"]) != response["total"]:
    print(f"WARNING: got {len(response['data'])} of {response['total']} records")
else:
    print(f"all {response['total']} record(s) present")

In [ ]:
# 2. a flat list of dict, customers only
customers = []
for record in response["data"]:
    customers.append({
        "id": record["id"],
        "name": record["name"],
        "city": record["city"],
        "order_count": len(record["orders"]),
    })

for row in customers:
    print(row)

In [ ]:
# 3. dict of dict lookup
by_id = {}
for row in customers:
    by_id[row["id"]] = row

print(by_id[103])
print("Chi's city:", by_id[103]["city"])

In [ ]:
# 4. dict of list, columnar
cols = {"id": [], "name": [], "city": []}
for row in customers:
    cols["id"].append(row["id"])
    cols["name"].append(row["name"])
    cols["city"].append(row["city"])

print(cols["name"])
print()
print("second row, pulled from each column:")
print(cols["id"][1], cols["name"][1], cols["city"][1])

In [ ]:
# 5. list of list, values only, fixed column order
plain = []
for row in customers:
    values = []
    for column in ["id", "name", "city"]:
        values.append(row[column])
    plain.append(values)

for values in plain:
    print(values)

In [ ]:
# 6. flatten to one row per order
orders = []
for record in response["data"]:
    for order in record["orders"]:
        orders.append({
            "customer_id": record["id"],
            "customer_name": record["name"],
            "city": record["city"],
            "item": order["item"],
            "amt": order["amt"],
        })

for row in orders:
    print(row)

print()
print(f"{len(response['data'])} customers -> {len(orders)} order rows")
print("Musa has an empty orders list, so the inner loop never ran. He is missing.")

In [ ]:
# 7. flatten again, keeping customers with no orders
orders_with_all = []
for record in response["data"]:
    if len(record["orders"]) == 0:
        orders_with_all.append({
            "customer_id": record["id"],
            "customer_name": record["name"],
            "city": record["city"],
            "item": None,
            "amt": None,
        })
    else:
        for order in record["orders"]:
            orders_with_all.append({
                "customer_id": record["id"],
                "customer_name": record["name"],
                "city": record["city"],
                "item": order["item"],
                "amt": order["amt"],
            })

print(f"{len(orders_with_all)} rows, Musa included")
for row in orders_with_all:
    print(row["customer_name"], row["item"], row["amt"])

In [ ]:
# 8. totals per customer, then per city
print("per customer:")
spend_by_customer = {}
for record in response["data"]:
    total = 0
    for order in record["orders"]:
        total = total + order["amt"]
    spend_by_customer[record["name"]] = total
    print(f"  {record['name']}: {len(record['orders'])} order(s), {total:,}")

print()
print("per city:")
by_city = {}
for row in orders:
    city = row["city"]
    if city in by_city:
        by_city[city] = by_city[city] + row["amt"]
    else:
        by_city[city] = row["amt"]

for city, total in by_city.items():
    print(f"  {city}: {total:,}")

In [ ]:
# 9. shortcuts: rank by spend
ranked = sorted(spend_by_customer.items(), key=lambda pair: pair[1], reverse=True)

for position, pair in enumerate(ranked[:3], start=1):
    name, total = pair
    print(f"{position}. {name}: {total:,}")

In [ ]:
# 10. shortcuts: the single biggest order
biggest = max(orders, key=lambda row: row["amt"])

print(f"{biggest['customer_name']} bought a {biggest['item']} for {biggest['amt']:,}")

In [ ]:
# 11. pandas, and check the totals agree
import pandas as pd

customers_df = pd.DataFrame(customers)
orders_df = pd.DataFrame(orders)

print(customers_df)
print()
print(orders_df)
print()

loop_total = 0
for row in orders:
    loop_total = loop_total + row["amt"]

print(f"loop total:   {loop_total:,}")
print(f"pandas total: {orders_df['amt'].sum():,}")
print("match" if loop_total == orders_df["amt"].sum() else "MISMATCH")

In [ ]:
# Stretch 12: as a reusable function
def to_rows(response, keep_empty=False):
    """Flatten an API response into one row per order."""
    out = []
    for record in response["data"]:
        parent = {
            "customer_id": record["id"],
            "customer_name": record["name"],
            "city": record["city"],
        }
        if len(record["orders"]) == 0:
            if keep_empty:
                out.append({**parent, "item": None, "amt": None})
            continue
        for order in record["orders"]:
            out.append({**parent, "item": order["item"], "amt": order["amt"]})
    return out


print(f"without empties: {len(to_rows(response))} rows")
print(f"with empties:    {len(to_rows(response, keep_empty=True))} rows")

In [ ]:
# Stretch 13: a reusable group-and-sum
def group_sum(rows, group_key, value_key):
    """Return a dict of group -> summed value."""
    totals = {}
    for row in rows:
        key = row[group_key]
        if key in totals:
            totals[key] = totals[key] + row[value_key]
        else:
            totals[key] = row[value_key]
    return totals


print("by city: ", group_sum(orders, "city", "amt"))
print("by item: ", group_sum(orders, "item", "amt"))
print("by name: ", group_sum(orders, "customer_name", "amt"))

In [ ]:
# Stretch 14: the same flatten with json_normalize
normalised = pd.json_normalize(
    response["data"], record_path="orders", meta=["id", "name", "city"])

print(normalised)
print()
print(f"loop version:          {len(orders)} rows")
print(f"json_normalize:        {len(normalised)} rows")
print("Both drop Musa, because record_path has nothing to walk for him.")
print(f"totals agree: {normalised['amt'].sum() == loop_total}")